In [7]:
# --- BLOK 1: IMPORT LIBRARY ---
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, ReLU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("Versi TensorFlow:", tf.__version__)

Versi TensorFlow: 2.17.0


In [8]:
# ============================================================
# BLOK 2: FEATURE ENGINEERING
# ============================================================

DATASET_NAME = "Primary"
DATASET_PATH = Path("Train_Test_Data.csv")

OUTPUT_ROOT = Path("deployment_artifacts")
DATASET_OUT = OUTPUT_ROOT / DATASET_NAME

RANDOM_STATE = 42
THRESHOLD = 0.50

FEATURES_ESP32 = [
    "temperature",
    "pressure",
    "humidity",
    "temp_diff",
    "press_diff",
    "hum_diff",
    "temp_roll_std",
    "temp_mean_5",
    "hum_mean_5",
]

print("================================================")
print("PRIMARY DATASET - ESP32 DNN/MLP")
print("================================================")
print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {DATASET_OUT}")
print()

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_PATH.resolve()}")

DATASET_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Memuat dataset...")
df = pd.read_csv(DATASET_PATH)

required_columns = [
    "temperature",
    "pressure",
    "humidity",
    "label_binary",
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise RuntimeError("Kolom wajib tidak ditemukan: " + ", ".join(missing_columns))

# ------------------------------------------------------------
# Feature engineering
# ------------------------------------------------------------
df["temp_diff"] = df["temperature"].diff().fillna(0)
df["press_diff"] = df["pressure"].diff().fillna(0)
df["hum_diff"] = df["humidity"].diff().fillna(0)
df["temp_roll_std"] = df["temperature"].rolling(5).std().fillna(0)
df["temp_mean_5"] = df["temperature"].rolling(5).mean().fillna(0)
df["hum_mean_5"] = df["humidity"].rolling(5).mean().fillna(0)

df = (
    df.replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Binary label mapping
# ------------------------------------------------------------
def encode_binary_labels(series):
    normalized = series.astype(str).str.strip().str.lower()

    normal_aliases = {
        "normal",
        "benign",
        "0",
    }

    attack_aliases = {
        "attack",
        "anomaly",
        "abnormal",
        "1",
    }

    unique_values = set(normalized.unique())

    if unique_values.issubset(normal_aliases | attack_aliases):
        y = normalized.map(lambda value: 0 if value in normal_aliases else 1)

        return (
            y.astype(np.int32),
            ["Normal", "Attack"],
        )

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    if numeric.notna().all():
        unique_numeric = set(numeric.astype(int).unique())

        if unique_numeric.issubset({0, 1}):
            return (
                numeric.astype(np.int32),
                ["Normal", "Attack"],
            )

    raise ValueError(
        "label_binary tidak dapat dipetakan "
        "secara aman ke 0=Normal dan 1=Attack.\n"
        f"Nilai ditemukan: "
        f"{sorted(map(str, series.unique()))}"
    )

y_series, CLASS_NAMES = encode_binary_labels(df["label_binary"])
X = df[FEATURES_ESP32].to_numpy(dtype=np.float32)
y = y_series.to_numpy(dtype=np.int32)

print("Feature list:")
for i, feature in enumerate(
    FEATURES_ESP32,
    start=1,
):
    print(f"  {i}. {feature}")

print()
print("Class mapping:")
print("  0 = Normal")
print("  1 = Attack")

print()
print("Dataset distribution:")
print(
    pd.Series(y)
    .map(
        {
            0: "Normal",
            1: "Attack",
        }
    )
    .value_counts()
)

print()
print(f"Total samples after FE : {len(df)}")

PRIMARY DATASET - ESP32 DNN/MLP
Dataset : Train_Test_Data.csv
Output  : deployment_artifacts\Primary

Memuat dataset...
Feature list:
  1. temperature
  2. pressure
  3. humidity
  4. temp_diff
  5. press_diff
  6. hum_diff
  7. temp_roll_std
  8. temp_mean_5
  9. hum_mean_5

Class mapping:
  0 = Normal
  1 = Attack

Dataset distribution:
Normal    182564
Attack    181900
Name: count, dtype: int64

Total samples after FE : 364464


In [9]:
# ============================================================
# BLOK 3: TRAIN, VALIDATION, TEST, EVALUATION
# ============================================================

# ------------------------------------------------------------
# Stratified split 80:10:10
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print()
print("================================================")
print("DATA SPLIT")
print("================================================")
print(f"Training   : {len(X_train)}")
print(f"Validation : {len(X_val)}")
print(f"Testing    : {len(X_test)}")


# ------------------------------------------------------------
# StandardScaler
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)


# ------------------------------------------------------------
# DNN / MLP model
# ------------------------------------------------------------

model_esp32 = Sequential(
    [
        Input(shape=(len(FEATURES_ESP32),)),
        Dense(
            32,
            kernel_regularizer=regularizers.l2(0.001),
        ),
        ReLU(),
        Dense(
            16,
            kernel_regularizer=regularizers.l2(0.001),
        ),
        ReLU(),
        Dense(1),
    ]
)

model_esp32.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

print()
print("================================================")
print("MODEL")
print("================================================")

model_esp32.summary()

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=0.00001,
    verbose=1,
)

history = model_esp32.fit(
    X_train_scaled,
    y_train,
    validation_data=(
        X_val_scaled,
        y_val,
    ),
    epochs=50,
    batch_size=256,
    callbacks=[
        early_stop,
        reduce_lr,
    ],
    verbose=1,
)

# ------------------------------------------------------------
# Float32 test evaluation
# ------------------------------------------------------------

test_logits = model_esp32.predict(
    X_test_scaled,
    verbose=0,
).reshape(-1)

test_probability = 1.0 / (1.0 + np.exp(-test_logits))

y_pred = (test_probability >= THRESHOLD).astype(np.int32)

accuracy = accuracy_score(
    y_test,
    y_pred,
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1],
)

tn, fp, fn, tp = cm.ravel()

far = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print()
print("================================================")
print("FLOAT32 TEST EVALUATION")
print("================================================")

print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

print(f"Accuracy        : " f"{accuracy * 100:.4f}%")
print(f"Precision Macro : " f"{precision * 100:.4f}%")
print(f"Recall Macro    : " f"{recall * 100:.4f}%")
print(f"F1-Score Macro  : " f"{f1 * 100:.4f}%")
print(f"False Alarm Rate: " f"{far * 100:.4f}%")

print()
print("Confusion Matrix")
print("                 Predicted")
print("                 Normal Attack")
print(f"Actual Normal    " f"{tn:6d} {fp:6d}")
print(f"Actual Attack    " f"{fn:6d} {tp:6d}")


DATA SPLIT
Training   : 291571
Validation : 36446
Testing    : 36447

MODEL


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 865 (3.38 KB)

 Trainable params: 865 (3.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7591 - loss: 0.5035 - val_accuracy: 0.8417 - val_loss: 0.3997 - learning_rate: 0.0010
Epoch 2/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8424 - loss: 0.3944 - val_accuracy: 0.8421 - val_loss: 0.3874 - learning_rate: 0.0010
Epoch 3/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 1s 907us/step - accuracy: 0.8421 - loss: 0.3863 - val_accuracy: 0.8424 - val_loss: 0.3839 - learning_rate: 0.0010
Epoch 4/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 1s 979us/step - accuracy: 0.8441 - loss: 0.3802 - val_accuracy: 0.8435 - val_loss: 0.3790 - learning_rate: 0.0010
Epoch 5/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8428 - loss: 0.3781 - val_accuracy: 0.8431 - val_loss: 0.3758 - learning_rate: 0.0010
Epoch 6/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 1s 864us/step - accuracy: 0.8439 - loss: 0.3752 - val_accuracy: 0.8444 - val_loss: 0.3732 - learning_rate: 0.0010
Epoch 7/50
1139/1139 ━━━━━━━━━━━━━━━━━━━━ 1s 888us/step - accuracy: 0.

In [10]:
# ============================================================
# BLOK 4: SAVE FILE-FILE ARTIFACT
# ============================================================
print()
print("================================================")
print("SAVING ARTIFACTS")
print("================================================")

# ------------------------------------------------------------
# 4.1 Save Float32 model
# ------------------------------------------------------------
float32_path = DATASET_OUT / "model_esp32_float32.keras"
model_esp32.save(float32_path)


# ------------------------------------------------------------
# 4.2 Representative dataset untuk INT8
# ------------------------------------------------------------
def representative_dataset():
    n = min(
        500,
        len(X_train_scaled),
    )

    indices = np.linspace(
        0,
        len(X_train_scaled) - 1,
        n,
        dtype=int,
    )

    for index in indices:
        yield [X_train_scaled[index : index + 1].astype(np.float32)]


# ------------------------------------------------------------
# 4.3 Convert Float32 -> TFLite INT8
# ------------------------------------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(model_esp32)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print()
print("Converting TFLite INT8...")

tflite_model = converter.convert()
tflite_path = DATASET_OUT / "model_esp32_int8.tflite"
tflite_path.write_bytes(tflite_model)

# ------------------------------------------------------------
# 4.4 Read actual INT8 quantization parameters
# ------------------------------------------------------------
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]

output_detail = interpreter.get_output_details()[0]
input_scale, input_zero_point = input_detail["quantization"]
output_scale, output_zero_point = output_detail["quantization"]

if input_scale == 0:
    raise RuntimeError("Input quantization scale = 0.")

if output_scale == 0:
    raise RuntimeError("Output quantization scale = 0.")

print()
print("INT8 Quantization:")
print(f"Input scale       : " f"{input_scale}")
print(f"Input zero point  : " f"{input_zero_point}")
print(f"Output scale      : " f"{output_scale}")
print(f"Output zero point : " f"{output_zero_point}")


# ------------------------------------------------------------
# 4.5 Save model as C header
# ------------------------------------------------------------
def write_model_header(
    model_bytes,
    output_path,
    variable_name="g_model",
):

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:

        file.write("#pragma once\n")
        file.write("#include <stdint.h>\n\n")
        file.write("alignas(16) " f"const unsigned char " f"{variable_name}[] = {{\n")

        for i in range(
            0,
            len(model_bytes),
            12,
        ):

            chunk = model_bytes[i : i + 12]

            file.write("  " + ", ".join(f"0x{byte:02x}" for byte in chunk) + ",\n")

        file.write("};\n\n")

        file.write(
            f"const unsigned int " f"{variable_name}_len = " f"{len(model_bytes)};\n"
        )


model_header_path = DATASET_OUT / "model_esp32_int8.h"

write_model_header(
    tflite_model,
    model_header_path,
)

# ------------------------------------------------------------
# 4.6 Save StandardScaler artifact
# ------------------------------------------------------------
scaler_npz_path = DATASET_OUT / "esp32_scaler.npz"

np.savez(
    scaler_npz_path,
    mean_=scaler.mean_.astype(np.float32),
    scale_=scaler.scale_.astype(np.float32),
)


def write_scaler_header(
    mean,
    scale,
    output_path,
):
    n_features = len(mean)

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:

        file.write("#pragma once\n\n")
        file.write(f"const float " f"ESP32_SCALER_MEAN" f"[{n_features}] = {{\n")
        file.write(
            "  " + ", ".join(f"{float(value):.10g}f" for value in mean) + "\n};\n\n"
        )
        file.write(f"const float " f"ESP32_SCALER_SCALE" f"[{n_features}] = {{\n")
        file.write(
            "  " + ", ".join(f"{float(value):.10g}f" for value in scale) + "\n};\n"
        )


scaler_header_path = DATASET_OUT / "esp32_scaler.h"

write_scaler_header(
    scaler.mean_,
    scaler.scale_,
    scaler_header_path,
)

# ------------------------------------------------------------
# 4.7 Save INT8 model configuration header
# ------------------------------------------------------------
model_config_path = DATASET_OUT / "esp32_model_config.h"

with open(
    model_config_path,
    "w",
    encoding="utf-8",
) as file:

    file.write("#pragma once\n\n")
    file.write(
        "constexpr float " "ESP32_INPUT_SCALE = " f"{float(input_scale):.12g}f;\n"
    )
    file.write(
        "constexpr int " "ESP32_INPUT_ZERO_POINT = " f"{int(input_zero_point)};\n"
    )
    file.write(
        "constexpr float " "ESP32_OUTPUT_SCALE = " f"{float(output_scale):.12g}f;\n"
    )
    file.write(
        "constexpr int " "ESP32_OUTPUT_ZERO_POINT = " f"{int(output_zero_point)};\n"
    )

# ------------------------------------------------------------
# 4.8 Create replay CSV
#
# Replay menggunakan raw engineered features.
# StandardScaler dilakukan kembali pada PC/ESP32
# menggunakan artifact scaler yang sama.
#
# Replay menggunakan TEST SET agar evaluasi deployment
# dilakukan terhadap data yang tidak digunakan untuk
# training maupun validation.
# ------------------------------------------------------------
replay_path = DATASET_OUT / "esp32_replay.csv"
replay_rows = []

for index in range(len(X_test)):
    replay_rows.append(
        {
            "sample_id": index,
            "temperature": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("temperature"),
                ]
            ),
            "pressure": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("pressure"),
                ]
            ),
            "humidity": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("humidity"),
                ]
            ),
            "temp_diff": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("temp_diff"),
                ]
            ),
            "press_diff": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("press_diff"),
                ]
            ),
            "hum_diff": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("hum_diff"),
                ]
            ),
            "temp_roll_std": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("temp_roll_std"),
                ]
            ),
            "temp_mean_5": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("temp_mean_5"),
                ]
            ),
            "hum_mean_5": float(
                X_test[
                    index,
                    FEATURES_ESP32.index("hum_mean_5"),
                ]
            ),
            "true_label": int(y_test[index]),
            "true_class": CLASS_NAMES[int(y_test[index])],
        }
    )

replay_df = pd.DataFrame(replay_rows)

replay_df.to_csv(
    replay_path,
    index=False,
)

# ------------------------------------------------------------
# 4.9 Save metadata
# ------------------------------------------------------------
metadata = {
    "dataset": DATASET_NAME,
    "dataset_file": DATASET_PATH.name,
    "model_type": "DNN_MLP",
    "framework": "TensorFlow/Keras",
    "tensorflow_version": tf.__version__,
    "features": FEATURES_ESP32,
    "class_names": CLASS_NAMES,
    "class_mapping": {
        "0": "Normal",
        "1": "Attack",
    },
    "threshold": THRESHOLD,
    "far_definition": "FP / (FP + TN)",
    "split": "80:10:10",
    "random_state": RANDOM_STATE,
    "train_samples": len(X_train),
    "validation_samples": len(X_val),
    "test_samples": len(X_test),
    "standard_scaler_mean": scaler.mean_.tolist(),
    "standard_scaler_scale": scaler.scale_.tolist(),
    "tflite_input_quantization": {
        "scale": float(input_scale),
        "zero_point": int(input_zero_point),
    },
    "tflite_output_quantization": {
        "scale": float(output_scale),
        "zero_point": int(output_zero_point),
    },
    "artifacts": {
        "float32_model": "model_esp32_float32.keras",
        "tflite_int8": "model_esp32_int8.tflite",
        "model_header": "model_esp32_int8.h",
        "scaler_npz": "esp32_scaler.npz",
        "scaler_header": "esp32_scaler.h",
        "model_config_header": "esp32_model_config.h",
        "replay_csv": "esp32_replay.csv",
    },
    "float32_test_metrics": {
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "far": float(far),
    },
    "float32_confusion_matrix": {
        "labels": [
            "Normal",
            "Attack",
        ],
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "matrix": cm.tolist(),
    },
}

metadata_path = DATASET_OUT / "esp32_metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metadata,
        file,
        indent=2,
    )

# ------------------------------------------------------------
# 4.10 Save training history
# ------------------------------------------------------------
history_path = DATASET_OUT / "training_history.csv"

pd.DataFrame(history.history).to_csv(
    history_path,
    index=False,
)

# ------------------------------------------------------------
# 4.11 Save test predictions
# ------------------------------------------------------------
test_predictions_path = DATASET_OUT / "float32_test_predictions.csv"

test_result_df = pd.DataFrame(
    {
        "true_label": y_test,
        "true_class": [CLASS_NAMES[int(label)] for label in y_test],
        "probability_attack": test_probability,
        "predicted_label": y_pred,
        "predicted_class": [CLASS_NAMES[int(label)] for label in y_pred],
    }
)

test_result_df.to_csv(
    test_predictions_path,
    index=False,
)

# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------
print()
print("================================================")
print("ARTIFACT SAVING COMPLETE")
print("================================================")

print()
print("Dataset:")
print(f"  {DATASET_PATH}")

print()
print("Artifacts:")

for path in [
    float32_path,
    tflite_path,
    model_header_path,
    scaler_npz_path,
    scaler_header_path,
    model_config_path,
    replay_path,
    metadata_path,
    history_path,
    test_predictions_path,
]:

    print(f"  {path}")

print()
print("Class mapping:")
print("  0 = Normal")
print("  1 = Attack")

print()
print("Replay source:")
print("  TEST SET ONLY")

print()
print("================================================")


SAVING ARTIFACTS

Converting TFLite INT8...
INFO:tensorflow:Assets written to: C:\Users\Rave\AppData\Local\Temp\tmpku1nwv3e\assets


INFO:tensorflow:Assets written to: C:\Users\Rave\AppData\Local\Temp\tmpku1nwv3e\assets


Saved artifact at 'C:\Users\Rave\AppData\Local\Temp\tmpku1nwv3e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 9), dtype=tf.float32, name='keras_tensor_12')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  3094770455424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3094770458064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3095701920176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3095701927040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3095701915248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3095701926688: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\Rave\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\convert.py:983: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 Quantization:
Input scale       : 0.18284697830677032
Input zero point  : 69
Output scale      : 0.7357223033905029
Output zero point : -124

ARTIFACT SAVING COMPLETE

Dataset:
  Train_Test_Data.csv

Artifacts:
  deployment_artifacts\Primary\model_esp32_float32.keras
  deployment_artifacts\Primary\model_esp32_int8.tflite
  deployment_artifacts\Primary\model_esp32_int8.h
  deployment_artifacts\Primary\esp32_scaler.npz
  deployment_artifacts\Primary\esp32_scaler.h
  deployment_artifacts\Primary\esp32_model_config.h
  deployment_artifacts\Primary\esp32_replay.csv
  deployment_artifacts\Primary\esp32_metadata.json
  deployment_artifacts\Primary\training_history.csv
  deployment_artifacts\Primary\float32_test_predictions.csv

Class mapping:
  0 = Normal
  1 = Attack

Replay source:
  TEST SET ONLY

